# DBSCAN Clustering with PyCaret

## Assignment (d): DBSCAN Clustering using PyCaret Library

**Author:** Nitish  
**Date:** December 2024

---

## Table of Contents
1. [Introduction](#introduction)
2. [DBSCAN Theory](#theory)
3. [PyCaret Setup](#setup)
4. [Data Preparation](#data)
5. [DBSCAN with PyCaret](#dbscan)
6. [Comparing Multiple Clustering Models](#comparison)
7. [Parameter Tuning](#tuning)
8. [Clustering Quality Metrics](#metrics)
9. [Visualization](#visualization)
10. [Conclusion](#conclusion)

---

<a id='introduction'></a>
## 1. Introduction

**DBSCAN (Density-Based Spatial Clustering of Applications with Noise)** is a density-based clustering algorithm that:
- Discovers clusters of arbitrary shape
- Automatically detects outliers/noise
- Doesn't require specifying number of clusters

**PyCaret** is a low-code machine learning library that automates ML workflows.

In [ ]:
# Install required packages
!pip install pycaret[full] numpy pandas matplotlib seaborn scikit-learn plotly -q

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons, make_circles, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.neighbors import NearestNeighbors
import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

# Set random seed
np.random.seed(42)

print("Base libraries imported successfully!")

<a id='theory'></a>
## 2. DBSCAN Theory

### Key Concepts:

1. **Core Points:** Points with at least `min_samples` neighbors within `eps` radius
2. **Border Points:** Points within `eps` of a core point but with fewer than `min_samples` neighbors
3. **Noise Points:** Points that are neither core nor border points

### Parameters:
- **eps (ε):** Maximum distance between two points to be considered neighbors
- **min_samples:** Minimum number of points to form a dense region (core point)

### Algorithm:
1. For each point, find all neighbors within eps
2. If point has ≥ min_samples neighbors, it's a core point
3. Create clusters by connecting core points that are neighbors
4. Assign border points to nearest core point's cluster
5. Mark remaining points as noise (-1)

### Advantages:
- No need to specify number of clusters
- Can find arbitrarily shaped clusters
- Robust to outliers

### Disadvantages:
- Sensitive to eps and min_samples
- Struggles with varying density clusters
- Not suitable for high-dimensional data

<a id='setup'></a>
## 3. PyCaret Setup

In [ ]:
# Import PyCaret clustering module
from pycaret.clustering import *

print("PyCaret clustering module imported successfully!")

<a id='data'></a>
## 4. Data Preparation

In [ ]:
# Generate synthetic datasets

# Dataset 1: Moon-shaped clusters (DBSCAN excels here)
X_moons, y_moons = make_moons(n_samples=500, noise=0.05, random_state=42)
df_moons = pd.DataFrame(X_moons, columns=['Feature_1', 'Feature_2'])
df_moons['True_Label'] = y_moons

# Dataset 2: Concentric circles
X_circles, y_circles = make_circles(n_samples=500, noise=0.05, factor=0.5, random_state=42)
df_circles = pd.DataFrame(X_circles, columns=['Feature_1', 'Feature_2'])
df_circles['True_Label'] = y_circles

# Dataset 3: Blobs with noise
X_blobs, y_blobs = make_blobs(n_samples=400, centers=3, cluster_std=0.5, random_state=42)
# Add noise points
noise = np.random.uniform(-10, 10, (50, 2))
X_noisy = np.vstack([X_blobs, noise])
y_noisy = np.concatenate([y_blobs, [-1] * 50])  # -1 for noise
df_noisy = pd.DataFrame(X_noisy, columns=['Feature_1', 'Feature_2'])
df_noisy['True_Label'] = y_noisy

print("Datasets created:")
print(f"  - Moons: {df_moons.shape}")
print(f"  - Circles: {df_circles.shape}")
print(f"  - Noisy Blobs: {df_noisy.shape}")

In [ ]:
# Visualize datasets
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

ax1 = axes[0]
scatter1 = ax1.scatter(df_moons['Feature_1'], df_moons['Feature_2'], 
                       c=df_moons['True_Label'], cmap='viridis', alpha=0.7, s=50)
ax1.set_title('Moon Dataset', fontsize=14)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')

ax2 = axes[1]
scatter2 = ax2.scatter(df_circles['Feature_1'], df_circles['Feature_2'], 
                       c=df_circles['True_Label'], cmap='viridis', alpha=0.7, s=50)
ax2.set_title('Circles Dataset', fontsize=14)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')

ax3 = axes[2]
colors = ['red' if l == -1 else plt.cm.viridis(l/3) for l in df_noisy['True_Label']]
ax3.scatter(df_noisy['Feature_1'], df_noisy['Feature_2'], c=colors, alpha=0.7, s=50)
ax3.set_title('Noisy Blobs Dataset\n(Red = Noise)', fontsize=14)
ax3.set_xlabel('Feature 1')
ax3.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

In [ ]:
# Load a real-world dataset for PyCaret
# Using Iris dataset
iris = load_iris()
df_iris = pd.DataFrame(iris.data, columns=iris.feature_names)
df_iris['species'] = iris.target

print("\nIris Dataset:")
print(df_iris.head())
print(f"\nShape: {df_iris.shape}")

<a id='dbscan'></a>
## 5. DBSCAN with PyCaret

In [ ]:
# Initialize PyCaret clustering setup for Moons dataset
# Remove true labels for unsupervised learning
df_moons_cluster = df_moons.drop('True_Label', axis=1)

# Setup PyCaret
cluster_setup = setup(data=df_moons_cluster, 
                      normalize=True,
                      session_id=42,
                      verbose=False)

In [ ]:
# Create DBSCAN model using PyCaret
dbscan_model = create_model('dbscan', eps=0.3, min_samples=5)

print("\nDBSCAN Model created successfully!")
print(dbscan_model)

In [ ]:
# Assign cluster labels to the data
dbscan_results = assign_model(dbscan_model)

print("\nClustering Results:")
print(dbscan_results.head(10))
print(f"\nCluster distribution:")
print(dbscan_results['Cluster'].value_counts())

In [ ]:
# Visualize DBSCAN results using PyCaret
plot_model(dbscan_model, plot='cluster')

In [ ]:
# Distribution plot
plot_model(dbscan_model, plot='distribution')

In [ ]:
# Compare with true labels
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax1 = axes[0]
scatter1 = ax1.scatter(df_moons['Feature_1'], df_moons['Feature_2'], 
                       c=df_moons['True_Label'], cmap='viridis', alpha=0.7, s=50)
ax1.set_title('True Labels', fontsize=14)
ax1.set_xlabel('Feature 1')
ax1.set_ylabel('Feature 2')
plt.colorbar(scatter1, ax=ax1, label='Cluster')

ax2 = axes[1]
# Map cluster labels to colors (handle noise as -1)
cluster_labels = dbscan_results['Cluster'].map(lambda x: -1 if 'Noise' in str(x) else int(str(x).replace('Cluster ', '')))
colors = ['red' if l == -1 else plt.cm.viridis(l/max(1, cluster_labels.max())) for l in cluster_labels]
ax2.scatter(dbscan_results['Feature_1'], dbscan_results['Feature_2'], c=colors, alpha=0.7, s=50)
ax2.set_title('DBSCAN Clustering (PyCaret)\n(Red = Noise)', fontsize=14)
ax2.set_xlabel('Feature 1')
ax2.set_ylabel('Feature 2')

plt.tight_layout()
plt.show()

<a id='comparison'></a>
## 6. Comparing Multiple Clustering Models with PyCaret

In [ ]:
# Setup for Iris dataset
df_iris_cluster = df_iris.drop('species', axis=1)

cluster_setup_iris = setup(data=df_iris_cluster, 
                           normalize=True,
                           session_id=42,
                           verbose=False)

In [ ]:
# Get list of available models
models()

In [ ]:
# Create multiple clustering models
print("Creating multiple clustering models...\n")

# K-Means
kmeans = create_model('kmeans', num_clusters=3)
print("K-Means created")

# DBSCAN
dbscan = create_model('dbscan', eps=0.5, min_samples=5)
print("DBSCAN created")

# Hierarchical
hierarchical = create_model('hclust', num_clusters=3)
print("Hierarchical created")

# Mean Shift
meanshift = create_model('meanshift')
print("Mean Shift created")

print("\nAll models created successfully!")

In [ ]:
# Compare models visually
models_dict = {
    'K-Means': kmeans,
    'DBSCAN': dbscan,
    'Hierarchical': hierarchical,
    'Mean Shift': meanshift
}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# True labels
ax = axes[0, 0]
scatter = ax.scatter(df_iris['sepal length (cm)'], df_iris['sepal width (cm)'], 
                    c=df_iris['species'], cmap='viridis', alpha=0.7, s=50)
ax.set_title('True Labels', fontsize=14)
ax.set_xlabel('Sepal Length')
ax.set_ylabel('Sepal Width')

# Plot each model
for idx, (name, model) in enumerate(models_dict.items()):
    results = assign_model(model)
    
    row = (idx + 1) // 3
    col = (idx + 1) % 3
    ax = axes[row, col]
    
    # Get numeric cluster labels
    try:
        cluster_labels = results['Cluster'].astype('category').cat.codes
    except:
        cluster_labels = results['Cluster']
    
    scatter = ax.scatter(results['sepal length (cm)'], results['sepal width (cm)'], 
                        c=cluster_labels, cmap='viridis', alpha=0.7, s=50)
    ax.set_title(f'{name}', fontsize=14)
    ax.set_xlabel('Sepal Length')
    ax.set_ylabel('Sepal Width')

# Hide empty subplot
axes[1, 2].axis('off')

plt.suptitle('Comparison of Clustering Algorithms on Iris Dataset', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Evaluate models
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score

print("=" * 70)
print("MODEL COMPARISON - IRIS DATASET")
print("=" * 70)

evaluation_results = []

for name, model in models_dict.items():
    results = assign_model(model)
    
    # Get numeric cluster labels
    try:
        labels = results['Cluster'].astype('category').cat.codes
    except:
        labels = results['Cluster']
    
    # Filter out noise for metrics calculation
    valid_mask = labels >= 0
    
    if valid_mask.sum() > 1 and len(np.unique(labels[valid_mask])) > 1:
        sil = silhouette_score(df_iris_cluster[valid_mask], labels[valid_mask])
        ch = calinski_harabasz_score(df_iris_cluster[valid_mask], labels[valid_mask])
        db = davies_bouldin_score(df_iris_cluster[valid_mask], labels[valid_mask])
        ari = adjusted_rand_score(df_iris['species'][valid_mask], labels[valid_mask])
        nmi = normalized_mutual_info_score(df_iris['species'][valid_mask], labels[valid_mask])
    else:
        sil = ch = db = ari = nmi = np.nan
    
    n_clusters = len(np.unique(labels[labels >= 0]))
    n_noise = (labels < 0).sum()
    
    evaluation_results.append({
        'Model': name,
        'Clusters': n_clusters,
        'Noise Points': n_noise,
        'Silhouette': sil,
        'Calinski-Harabasz': ch,
        'Davies-Bouldin': db,
        'ARI': ari,
        'NMI': nmi
    })

eval_df = pd.DataFrame(evaluation_results)
print(eval_df.to_string(index=False))

<a id='tuning'></a>
## 7. Parameter Tuning for DBSCAN

In [ ]:
# Method 1: K-distance graph for finding optimal eps
def find_optimal_eps(X, min_samples=5):
    """Use k-distance graph to find optimal eps"""
    neighbors = NearestNeighbors(n_neighbors=min_samples)
    neighbors.fit(X)
    distances, _ = neighbors.kneighbors(X)
    
    # Sort distances to k-th nearest neighbor
    k_distances = np.sort(distances[:, -1])
    
    return k_distances

# Apply to moons dataset
scaler = StandardScaler()
X_moons_scaled = scaler.fit_transform(df_moons[['Feature_1', 'Feature_2']])

k_distances = find_optimal_eps(X_moons_scaled, min_samples=5)

plt.figure(figsize=(10, 6))
plt.plot(range(len(k_distances)), k_distances, 'b-', linewidth=2)
plt.xlabel('Points (sorted by distance)', fontsize=12)
plt.ylabel('5-th Nearest Neighbor Distance', fontsize=12)
plt.title('K-Distance Graph for Optimal eps Selection', fontsize=14)
plt.grid(True, alpha=0.3)

# Find elbow point (approximate)
elbow_idx = np.argmax(np.diff(k_distances)) + 1
optimal_eps = k_distances[elbow_idx]
plt.axhline(y=optimal_eps, color='r', linestyle='--', label=f'Suggested eps ≈ {optimal_eps:.3f}')
plt.legend()
plt.tight_layout()
plt.show()

print(f"Suggested optimal eps: {optimal_eps:.3f}")

In [ ]:
# Method 2: Grid search for best parameters
def dbscan_grid_search(X, y_true, eps_range, min_samples_range):
    """Grid search for DBSCAN parameters"""
    results = []
    
    for eps in eps_range:
        for min_samples in min_samples_range:
            dbscan = DBSCAN(eps=eps, min_samples=min_samples)
            labels = dbscan.fit_predict(X)
            
            n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
            n_noise = list(labels).count(-1)
            
            # Calculate metrics only if we have valid clusters
            valid_mask = labels >= 0
            if valid_mask.sum() > 1 and n_clusters > 1:
                sil = silhouette_score(X[valid_mask], labels[valid_mask])
                ari = adjusted_rand_score(y_true[valid_mask], labels[valid_mask])
            else:
                sil = -1
                ari = -1
            
            results.append({
                'eps': eps,
                'min_samples': min_samples,
                'n_clusters': n_clusters,
                'n_noise': n_noise,
                'silhouette': sil,
                'ari': ari
            })
    
    return pd.DataFrame(results)

# Grid search on moons dataset
eps_range = np.arange(0.1, 0.6, 0.05)
min_samples_range = [3, 5, 7, 10]

grid_results = dbscan_grid_search(X_moons_scaled, y_moons, eps_range, min_samples_range)

print("Grid Search Results (Top 10 by Silhouette):")
print(grid_results.sort_values('silhouette', ascending=False).head(10).to_string(index=False))

In [ ]:
# Visualize parameter sensitivity
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

# Heatmap of silhouette scores
pivot_sil = grid_results.pivot(index='min_samples', columns='eps', values='silhouette')
ax1 = axes[0, 0]
sns.heatmap(pivot_sil, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax1)
ax1.set_title('Silhouette Score Heatmap', fontsize=14)

# Heatmap of number of clusters
pivot_clusters = grid_results.pivot(index='min_samples', columns='eps', values='n_clusters')
ax2 = axes[0, 1]
sns.heatmap(pivot_clusters, annot=True, fmt='d', cmap='Blues', ax=ax2)
ax2.set_title('Number of Clusters Heatmap', fontsize=14)

# Heatmap of noise points
pivot_noise = grid_results.pivot(index='min_samples', columns='eps', values='n_noise')
ax3 = axes[1, 0]
sns.heatmap(pivot_noise, annot=True, fmt='d', cmap='Reds', ax=ax3)
ax3.set_title('Number of Noise Points Heatmap', fontsize=14)

# Heatmap of ARI
pivot_ari = grid_results.pivot(index='min_samples', columns='eps', values='ari')
ax4 = axes[1, 1]
sns.heatmap(pivot_ari, annot=True, fmt='.2f', cmap='Greens', ax=ax4)
ax4.set_title('Adjusted Rand Index Heatmap', fontsize=14)

plt.suptitle('DBSCAN Parameter Sensitivity Analysis', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Visualize DBSCAN with different parameters
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

params_list = [
    (0.1, 5), (0.2, 5), (0.3, 5),
    (0.3, 3), (0.3, 7), (0.3, 10)
]

for idx, (eps, min_samples) in enumerate(params_list):
    row = idx // 3
    col = idx % 3
    ax = axes[row, col]
    
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X_moons_scaled)
    
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    # Color noise points differently
    colors = ['red' if l == -1 else plt.cm.viridis(l/max(1, labels.max())) for l in labels]
    ax.scatter(X_moons_scaled[:, 0], X_moons_scaled[:, 1], c=colors, alpha=0.7, s=40)
    ax.set_title(f'eps={eps}, min_samples={min_samples}\nClusters: {n_clusters}, Noise: {n_noise}', fontsize=12)
    ax.set_xlabel('Feature 1')
    ax.set_ylabel('Feature 2')

plt.suptitle('DBSCAN Parameter Sensitivity (Red = Noise)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

<a id='metrics'></a>
## 8. Clustering Quality Metrics

In [ ]:
def comprehensive_dbscan_evaluation(X, y_true, eps, min_samples, name="Dataset"):
    """Comprehensive DBSCAN evaluation"""
    dbscan = DBSCAN(eps=eps, min_samples=min_samples)
    labels = dbscan.fit_predict(X)
    
    print(f"\n{'='*60}")
    print(f"DBSCAN EVALUATION: {name}")
    print(f"Parameters: eps={eps}, min_samples={min_samples}")
    print(f"{'='*60}")
    
    # Cluster statistics
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise = list(labels).count(-1)
    
    print(f"\n--- Cluster Statistics ---")
    print(f"Number of clusters: {n_clusters}")
    print(f"Noise points: {n_noise} ({100*n_noise/len(labels):.1f}%)")
    
    # Cluster sizes
    unique, counts = np.unique(labels[labels >= 0], return_counts=True)
    for cluster, count in zip(unique, counts):
        print(f"Cluster {cluster}: {count} points")
    
    # Metrics (excluding noise)
    valid_mask = labels >= 0
    
    if valid_mask.sum() > 1 and n_clusters > 1:
        print(f"\n--- Internal Metrics (excluding noise) ---")
        print(f"Silhouette Score:        {silhouette_score(X[valid_mask], labels[valid_mask]):.4f}")
        print(f"Calinski-Harabasz Index: {calinski_harabasz_score(X[valid_mask], labels[valid_mask]):.4f}")
        print(f"Davies-Bouldin Index:    {davies_bouldin_score(X[valid_mask], labels[valid_mask]):.4f}")
        
        if y_true is not None:
            print(f"\n--- External Metrics ---")
            print(f"Adjusted Rand Index:     {adjusted_rand_score(y_true[valid_mask], labels[valid_mask]):.4f}")
            print(f"Normalized Mutual Info:  {normalized_mutual_info_score(y_true[valid_mask], labels[valid_mask]):.4f}")
    else:
        print("\nInsufficient clusters for metric calculation.")
    
    return labels

# Evaluate on different datasets
labels_moons = comprehensive_dbscan_evaluation(X_moons_scaled, y_moons, eps=0.3, min_samples=5, name="Moons")

In [ ]:
# Evaluate on circles dataset
X_circles_scaled = scaler.fit_transform(df_circles[['Feature_1', 'Feature_2']])
labels_circles = comprehensive_dbscan_evaluation(X_circles_scaled, y_circles, eps=0.2, min_samples=5, name="Circles")

In [ ]:
# Evaluate on noisy blobs dataset
X_noisy_scaled = scaler.fit_transform(df_noisy[['Feature_1', 'Feature_2']])
labels_noisy = comprehensive_dbscan_evaluation(X_noisy_scaled, y_noisy, eps=0.3, min_samples=5, name="Noisy Blobs")

<a id='visualization'></a>
## 9. Visualization

In [ ]:
# Final comparison visualization
fig, axes = plt.subplots(2, 3, figsize=(18, 12))

datasets = [
    ('Moons', X_moons_scaled, y_moons, labels_moons),
    ('Circles', X_circles_scaled, y_circles, labels_circles),
    ('Noisy Blobs', X_noisy_scaled, y_noisy, labels_noisy)
]

for idx, (name, X, y_true, labels_pred) in enumerate(datasets):
    # True labels
    ax1 = axes[0, idx]
    colors_true = ['gray' if l == -1 else plt.cm.viridis(l/max(1, y_true.max())) for l in y_true]
    ax1.scatter(X[:, 0], X[:, 1], c=colors_true, alpha=0.7, s=40)
    ax1.set_title(f'{name} - True Labels', fontsize=14)
    ax1.set_xlabel('Feature 1')
    ax1.set_ylabel('Feature 2')
    
    # DBSCAN labels
    ax2 = axes[1, idx]
    colors_pred = ['red' if l == -1 else plt.cm.viridis(l/max(1, labels_pred.max())) for l in labels_pred]
    ax2.scatter(X[:, 0], X[:, 1], c=colors_pred, alpha=0.7, s=40)
    n_clusters = len(set(labels_pred)) - (1 if -1 in labels_pred else 0)
    n_noise = list(labels_pred).count(-1)
    ax2.set_title(f'{name} - DBSCAN\nClusters: {n_clusters}, Noise: {n_noise}', fontsize=14)
    ax2.set_xlabel('Feature 1')
    ax2.set_ylabel('Feature 2')

plt.suptitle('DBSCAN Clustering Results (Red = Noise)', fontsize=16, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Interactive visualization with Plotly
df_viz = pd.DataFrame({
    'Feature_1': X_moons_scaled[:, 0],
    'Feature_2': X_moons_scaled[:, 1],
    'True_Label': y_moons,
    'DBSCAN_Label': labels_moons
})

df_viz['DBSCAN_Label_Str'] = df_viz['DBSCAN_Label'].apply(lambda x: 'Noise' if x == -1 else f'Cluster {x}')

fig = px.scatter(df_viz, x='Feature_1', y='Feature_2', color='DBSCAN_Label_Str',
                 title='DBSCAN Clustering on Moons Dataset (Interactive)',
                 color_discrete_map={'Noise': 'red'})
fig.update_layout(width=800, height=600)
fig.show()

<a id='conclusion'></a>
## 10. Conclusion

### Summary

In this notebook, we explored **DBSCAN clustering** using the **PyCaret** library.

### Key Findings:

1. **DBSCAN Strengths:**
   - Excellent for non-convex clusters (moons, circles)
   - Automatic outlier/noise detection
   - No need to specify number of clusters

2. **Parameter Selection:**
   - K-distance graph helps find optimal eps
   - min_samples typically set to 2*dimensions or higher
   - Grid search useful for fine-tuning

3. **PyCaret Benefits:**
   - Easy model creation and comparison
   - Built-in visualization
   - Automatic preprocessing

4. **Limitations:**
   - Sensitive to parameter selection
   - Struggles with varying density clusters
   - May not work well in high dimensions

### When to Use DBSCAN:
- Clusters have arbitrary shapes
- Data contains noise/outliers
- Number of clusters is unknown
- Clusters have similar density

### References:
- Ester, M., et al. (1996). "A density-based algorithm for discovering clusters"
- PyCaret Documentation: https://pycaret.org/

In [ ]:
# Final summary
print("=" * 70)
print("           DBSCAN CLUSTERING WITH PYCARET - FINAL SUMMARY")
print("=" * 70)
print("\n✓ Implemented DBSCAN using PyCaret library")
print("✓ Compared multiple clustering algorithms")
print("✓ Demonstrated parameter tuning with K-distance graph")
print("✓ Performed grid search for optimal parameters")
print("✓ Tested on various datasets (moons, circles, noisy blobs)")
print("✓ Evaluated using multiple clustering quality metrics")
print("✓ Visualized results with static and interactive plots")
print("\n" + "=" * 70)